In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import Imputer 
from pyspark.sql import Row

In [0]:
from_container = 'bronze'
storage_acc = 'adlsolistchurn2026'
read_url = f'abfss://{from_container}@{storage_acc}.dfs.core.windows.net/'

to_container = 'silver'
write_url = f'abfss://{to_container}@{storage_acc}.dfs.core.windows.net/'


![image_1788437845242.png](./image_1788437845242.png "image_1788437845242.png")

* Customers table and Sellers table contain zipcodes which should be set as string datatype

In [0]:
def report(df):
    df.show(5)
    print("*Schema*"*10)
    df.printSchema()
    print("*Distinct Count*"*10)
    print(df.count())
    print("*Null Count*"*10)
    print(df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).toPandas().to_string())


## 1. customer df

In [0]:
# 1
customer_schema = StructType([
    StructField('customer_id', StringType(), False),
    StructField('customer_unique_id', StringType(), True),
    StructField('customer_zip_code_prefix', StringType(), True),
    StructField('customer_city', StringType(), True),
    StructField('customer_state', StringType(), True)])
customer_df = spark.read.csv(read_url+'olist_customers_dataset.csv', header=True, schema=customer_schema)
report(customer_df)

In [0]:
# customer_df.dropDuplicates(subset=['customer_id']).count()

In [0]:
customer_df.write.format('delta').mode('overwrite').save(write_url+'olist_customers')

## 2. order_df

In [0]:
# 2
orders_df = spark.read.csv(read_url+'olist_orders_dataset.csv', header=True, inferSchema=True)
report(orders_df)

* There are Null records in the order table
* Not dropping missing values because customers purchase time is very crucial.

In [0]:
orders_df = orders_df.withColumn(
    'is_imputed',
    F.when(
        F.col('order_approved_at').isNull()
        | F.col('order_delivered_carrier_date').isNull()
        | F.col('order_delivered_customer_date').isNull(),
        F.lit(True)
    ).otherwise(F.lit(False))
)


In [0]:
from pyspark.ml import Transformer

class TimestampImputer(Transformer):
    def __init__(self, colA, colB):
        super(TimestampImputer, self).__init__()
        self.colA = colA
        self.colB = colB

    def _transform(self, df):
        avg_diff = df.select(F.mean(df[self.colA] - df[self.colB])).collect()[0][0]
        column = df.withColumn(
            self.colA,
            F.when(df[self.colA].isNull(), df[self.colB] + F.lit(avg_diff)).otherwise(df[self.colA])
        )
        return column


In [0]:

imputer = TimestampImputer('order_approved_at', "order_purchase_timestamp")
orders_df = imputer.transform(orders_df)
imputer.colA = 'order_delivered_carrier_date'
orders_df = imputer.transform(orders_df)
imputer.colA = 'order_delivered_customer_date'
orders_df = imputer.transform(orders_df)


In [0]:
report(orders_df)

In [0]:
orders_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(write_url+'olist_orders')

## 3. order_items_df

In [0]:
order_items_df = spark.read.csv(read_url+'olist_order_items_dataset.csv', header=True, inferSchema=True)
report(order_items_df)


In [0]:
order_items_df.describe().show()

In [0]:
order_items_df.write.format('delta').mode('overwrite').save(write_url+'olist_order_items')

## 4. order_review_df

In [0]:
# 4
order_review_df = spark.read.csv(read_url+'olist_order_reviews_dataset.csv', header=True, inferSchema=True)
report(order_review_df)

* Nulls are there in the order_review table
* review_score is set string

In [0]:
order_review_df = order_review_df.withColumn(
    'is_imputed',
    F.when(
        F.col('review_score').isNull()
        | F.col('review_comment_title').isNull()
        | F.col('review_comment_message').isNull()
        | F.col('review_creation_date').isNull(),
        F.lit(True)
    ).otherwise(F.lit(False))
)


In [0]:
display(order_review_df.describe())

In [0]:
order_review_df = order_review_df.drop_duplicates().dropna(subset=['order_id','review_id'])
#len(order_review_df.select(F.col("review_id")).collect()[5][0]) #the length of the review_id is 32
order_review_df = order_review_df.filter(F.length(F.col("review_id")) == 32)
mean_review_score = round(float(order_review_df.describe('review_score').collect()[1][1]),1)
order_review_df = order_review_df.withColumn(
                'review_score', 
                F.when(F.col("review_score").try_cast("float").isNotNull(),
                        F.col("review_score").cast("float")).otherwise(mean_review_score)) \
                .withColumn('review_comment_title', 
                        F.when(F.col('review_comment_title').isNull(), 
                                F.lit('No title')).otherwise(F.col('review_comment_title'))) \
                .withColumn('review_comment_message', 
                        F.when(F.col('review_comment_message').isNull(), 
                               F.lit('No comment')).otherwise(F.col('review_comment_message'))) \
                .withColumn('review_creation_date', 
                        F.when(F.col("review_creation_date").try_cast("timestamp").isNotNull(),
                                    F.col("review_creation_date").cast("timestamp"))
                                    .otherwise(F.lit("2018-03-29 00:00:00").cast("timestamp"))) \
                .withColumn('review_answer_timestamp', 
                        F.when(F.col("review_answer_timestamp").try_cast("timestamp").isNotNull(),
                               F.col("review_answer_timestamp"))
                            .otherwise(F.lit("Not_answered")))

In [0]:
report(order_review_df)

In [0]:
order_review_df.write.format('delta').mode('overwrite').option("overwriteSchema", "true").save(write_url+'olist_order_reviews')

## order_payment_df

In [0]:
# 5
order_payment_df = spark.read.csv(read_url+'olist_order_payments_dataset.csv', header=True, inferSchema=True)
report(order_payment_df)

In [0]:
order_payment_df.describe().show()

In [0]:
order_payment_df.select('payment_type').distinct().show()

In [0]:
order_payment_df.write.format('delta').mode('overwrite').save(write_url+'olist_order_payments')

## 6. product_df

In [0]:
# 6
product_df = spark.read.csv(read_url+'olist_products_dataset.csv', header=True, inferSchema=True)
report(product_df)

* There are NUlls in the product table
* some column names are incorrect

In [0]:
product_df = product_df.withColumn(
    'is_imputed',
    F.col('product_name_lenght').isNull()
    | F.col('product_description_lenght').isNull()
    | F.col('product_photos_qty').isNull()
    | F.col('product_weight_g').isNull()
    | F.col('product_length_cm').isNull()
    | F.col('product_height_cm').isNull()
    | F.col('product_width_cm').isNull()
    | F.col('product_category_name').isNull()
)

In [0]:
impute_cols = ['product_name_length', 'product_description_length',
                            'product_photos_qty', 'product_weight_g', 'product_length_cm',
                            'product_height_cm', 'product_width_cm']
mean_imputer = Imputer(strategy="mean")
mean_imputer.setInputCols(impute_cols)
mean_imputer.setOutputCols(impute_cols)

In [0]:
product_df = product_df.dropna(thresh=4) \
            .withColumnsRenamed({"product_description_lenght": "product_description_length",
                               "product_name_lenght": "product_name_length"}) \
            .withColumn('product_category_name', F.when(F.col('product_category_name').isNull(), 
                                F.lit('outro')).otherwise(F.col('product_category_name')))

product_df = mean_imputer.fit(product_df).transform(product_df)

In [0]:
#drop nulls more than 8 fileds
report(product_df)

In [0]:
product_df.write.format('delta').mode('overwrite').option("overwriteSchema", "true").save(write_url+'olist_products')

## 7. product_category_name_trans

In [0]:
# 7
product_category_name_trans_df = spark.read.csv(read_url+'product_category_name_translation.csv', header=True, inferSchema=True)
report(product_category_name_trans_df)

In [0]:
new_row = {'product_category_name': ['outro','pc_gamer','portateis_cozinha_e_preparadores_de_alimentos'], 'product_category_name_english': ['other','pc_gamer','portable_cookware_and_food_processors']}
product_category_name_trans_df = product_category_name_trans_df.unionByName(spark.createDataFrame([new_row]))


In [0]:
new_rows = [
    Row(product_category_name='outro', product_category_name_english='other'),
    Row(product_category_name='pc_gamer', product_category_name_english='pc_gamer'),
    Row(product_category_name='portateis_cozinha_e_preparadores_de_alimentos', product_category_name_english='portable_cookware_and_food_processors')
]
product_category_name_trans_df = product_category_name_trans_df.unionByName(spark.createDataFrame(new_rows))


In [0]:
product_category_name_trans_df.write.format('delta').mode('overwrite').save(write_url+'product_category_name_translation')

## 8. seller_df

In [0]:
# 8
seller_schema = StructType([
    StructField('seller_id', StringType(), False),
    StructField('seller_zip_code_prefix', StringType(), True),
    StructField('seller_city', StringType(), True),
    StructField('seller_state', StringType(), True)
])
seller_df = spark.read.csv(read_url+'olist_sellers_dataset.csv', header=True, schema=seller_schema)
report(seller_df)

In [0]:
seller_df.write.format('delta').mode('overwrite').save(write_url+'olist_sellers')

## 9. geolocation

In [0]:
# 9

geoloc_schema = StructType([
    StructField('geolocation_zip_code_prefix', StringType(), False),
    StructField('geolocation_lat', DoubleType(), True),
    StructField('geolocation_lng', DoubleType(), True),
    StructField('geolocation_city', StringType(), True),
    StructField('geolocation_state', StringType(), True)
])
geoloc_df = spark.read.csv(read_url+'olist_geolocation_dataset.csv', header=True, schema=geoloc_schema)
report(geoloc_df)

In [0]:
geoloc_df = geoloc_df.dropDuplicates(['geolocation_zip_code_prefix'])

In [0]:
report(geoloc_df)

In [0]:
geoloc_df.write.format('delta').mode('overwrite').save(write_url+'olist_geolocation')